In [1]:
# ============================================================
# Notebook : 06_ablation.ipynb
# Project  : Causal-Guided Multimodal Sensor Fusion for Elderly Risk Detection
# Purpose  : CONTROLLED ablation. Unlike the original study (which swapped the
#            whole architecture), here the architecture, training setup and
#            edge attributes are held FIXED and exactly ONE factor is varied:
#            (A) graph topology: causal / fully-connected / random / none
#            (B) lag edge attribute: on vs off (causal topology)
#            (C) modality: all / EM-only / SM-only (causal topology)
#            All runs use the same household-level Group 5-fold split and the
#            same training-only standardization. The "causal + lag + all"
#            configuration is the proposed model (reused from notebook 04).
# Note     : Anonymous repository for double-blind review.
# ============================================================

from pathlib import Path
import numpy as np, pandas as pd, random
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import f1_score, roc_auc_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(2)
EPOCHS = 25; LR = 1e-3; WD = 1e-4; BATCH = 256

ROOT     = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROC_DIR = ROOT / "data" / "processed"; TAB_DIR = ROOT / "results" / "tables"
TAB_DIR.mkdir(parents=True, exist_ok=True)

z = np.load(PROC_DIR / "windows.npz", allow_pickle=True)
X = z["X"].astype(np.float32); y = z["y"].astype(np.int64)
sample_fold = z["sample_fold"].astype(int)
FEAT = list(z["feat_cols"]); N = len(FEAT); WINDOW = int(z["window"])
EM_IDX = list(z["em_idx"]); SM_IDX = list(z["sm_idx"]); TAU_MAX = 6
N_FOLDS = int(sample_fold.max()) + 1

class ProposedModel(nn.Module):
    def __init__(self, mask, lag, use_lag=True, node_dim=32, heads=4, hid=64, dp=0.3):
        super().__init__()
        self.register_buffer("mask", torch.tensor(mask)); self.register_buffer("lag", torch.tensor(lag))
        self.use_lag = use_lag; self.h = heads; self.hd = hid // heads; self.hid = hid; self.n = mask.shape[0]
        self.temp = nn.Sequential(nn.Conv1d(1, 16, 3, padding=1), nn.ReLU(),
                                  nn.Conv1d(16, node_dim, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool1d(1))
        self.q = nn.Linear(node_dim, hid); self.k = nn.Linear(node_dim, hid); self.v = nn.Linear(node_dim, hid)
        self.lagw = nn.Parameter(torch.zeros(self.h)) if use_lag else None
        self.proj = nn.Linear(hid, hid); self.drop = nn.Dropout(dp)
        self.clf = nn.Sequential(nn.Linear(hid, 32), nn.ReLU(), nn.Dropout(dp), nn.Linear(32, 2))
    def forward(self, x):
        B = x.shape[0]; Nn = self.n
        h = self.temp(x.reshape(B * Nn, 1, x.shape[-1])).squeeze(-1).view(B, Nn, -1); h = self.drop(h)
        q = self.q(h).view(B, Nn, self.h, self.hd).transpose(1, 2)
        k = self.k(h).view(B, Nn, self.h, self.hd).transpose(1, 2)
        v = self.v(h).view(B, Nn, self.h, self.hd).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) / (self.hd ** 0.5)
        eye = torch.eye(Nn, device=x.device)[None, None]
        allow = ((self.mask[None, None] + eye) > 0).float()
        if self.use_lag: att = att + self.lagw.view(1, self.h, 1, 1) * self.lag[None, None]
        att = torch.nan_to_num(torch.softmax(att.masked_fill(allow == 0, float("-inf")), dim=-1))
        out = (att @ v).transpose(1, 2).contiguous().view(B, Nn, self.hid)
        out = self.drop(F.elu(self.proj(out)))
        return self.clf(out.mean(dim=1))

def standardize(Xtr_raw, Xall_raw):
    mu = Xtr_raw.mean(axis=(0, 2), keepdims=True); sd = Xtr_raw.std(axis=(0, 2), keepdims=True); sd[sd == 0] = 1.0
    return (Xall_raw - mu) / sd

def causal_masks(f, idx=None):
    adj = np.load(PROC_DIR / f"causal_adj_fold{f}.npy")
    ii = list(range(N)) if idx is None else list(idx)
    n = len(ii); mask = np.zeros((n, n), np.float32); lag = np.zeros((n, n), np.float32)
    for a, i in enumerate(ii):
        for b, j in enumerate(ii):
            taus = [t for t in range(1, adj.shape[2]) if adj[i, j, t] == 1]
            if taus: mask[b, a] = 1.0; lag[b, a] = float(min(taus))
    return mask, lag

def topo_masks(kind, f, n=N):
    if kind == "causal":
        return causal_masks(f)
    if kind == "fc":
        m = np.ones((n, n), np.float32); np.fill_diagonal(m, 0); return m, np.zeros((n, n), np.float32)
    if kind == "random":
        cm, _ = causal_masks(f); ne = int(cm.sum()); r = np.random.default_rng(SEED)
        m = np.zeros((n, n), np.float32); pairs = [(i, j) for i in range(n) for j in range(n) if i != j]
        for c in r.choice(len(pairs), size=min(ne, len(pairs)), replace=False): m[pairs[c]] = 1.0
        return m, np.zeros((n, n), np.float32)
    return np.zeros((n, n), np.float32), np.zeros((n, n), np.float32)  # none

def run(mask, lag, use_lag, Xsub, idx_tr, idx_va, idx_te, cw):
    def dl(idx, sh): return DataLoader(TensorDataset(torch.tensor(Xsub[idx]), torch.tensor(y[idx])), batch_size=BATCH, shuffle=sh)
    tr, va, te = dl(idx_tr, True), dl(idx_va, False), dl(idx_te, False)
    torch.manual_seed(SEED); model = ProposedModel(mask, lag, use_lag=use_lag)
    crit = nn.CrossEntropyLoss(weight=cw); opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS); best = -1; bs = None
    for ep in range(EPOCHS):
        model.train()
        for xb, yb in tr:
            loss = crit(model(xb), yb); opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        sch.step(); model.eval(); P = []; T = []
        with torch.no_grad():
            for xb, yb in va: P.extend(model(xb).argmax(1).numpy()); T.extend(yb.numpy())
        vf = f1_score(T, P, average="macro", zero_division=0)
        if vf > best: best = vf; bs = {k: v.clone() for k, v in model.state_dict().items()}
    model.load_state_dict(bs); model.eval(); P = []; T = []; PR = []
    with torch.no_grad():
        for xb, yb in te:
            lo = model(xb); P.extend(lo.argmax(1).numpy()); T.extend(yb.numpy()); PR.extend(F.softmax(lo, 1)[:, 1].numpy())
    return (f1_score(T, P, average="macro", zero_division=0),
            f1_score(T, P, average="binary", zero_division=0), roc_auc_score(T, PR))

# variants: (name, builder) ; builder returns (mask, lag, use_lag, Xsub_full_or_idx)
VARIANTS = [
    ("A: Fully-connected + lag", "fc", True, None),
    ("A: Random graph + lag",    "random", True, None),
    ("A: No inter-sensor + lag", "none", True, None),
    ("B: Causal + NO lag",       "causal", False, None),
    ("C: Causal + lag, EM-only", "causalEM", True, EM_IDX),
    ("C: Causal + lag, SM-only", "causalSM", True, SM_IDX),
]
rows = []
for f in range(N_FOLDS):
    va = (f + 1) % N_FOLDS
    idx_te = np.where(sample_fold == f)[0]; idx_va = np.where(sample_fold == va)[0]
    idx_tr = np.where((sample_fold != f) & (sample_fold != va))[0]
    Xs = standardize(X[idx_tr], X)
    cw = torch.tensor([1.0, (y[idx_tr] == 0).sum() / max((y[idx_tr] == 1).sum(), 1)], dtype=torch.float)
    for name, kind, use_lag, idx in VARIANTS:
        if kind.startswith("causal") and idx is not None:      # modality subset
            mask, lag = causal_masks(f, idx); Xsub = Xs[:, idx, :]
        elif kind == "causal":
            mask, lag = causal_masks(f); Xsub = Xs
        else:
            mask, lag = topo_masks(kind, f, N); Xsub = Xs
        mf, bf, auc = run(mask, lag, use_lag, Xsub, idx_tr, idx_va, idx_te, cw)
        rows.append({"variant": name, "fold": f, "macro_f1": round(mf, 4),
                     "binary_f1": round(bf, 4), "auc_roc": round(auc, 4)})
        print(f"fold {f} | {name:<28} macroF1={mf:.3f} AUC={auc:.3f}")

abl = pd.DataFrame(rows)
# attach the proposed (causal+lag+all) from notebook 04 for a complete table
prop = pd.read_csv(TAB_DIR / "cv_proposed.csv").rename(columns={"model": "variant"})
prop["variant"] = "Proposed: Causal + lag, all"
abl = pd.concat([abl, prop[["variant", "fold", "macro_f1", "binary_f1", "auc_roc"]]], ignore_index=True)
abl.to_csv(TAB_DIR / "cv_ablation.csv", index=False)
print("\nAblation summary (mean +/- std over folds):")
for v in abl.variant.unique():
    sub = abl[abl.variant == v]
    print(f"  {v:<30} AUC {sub.auc_roc.mean():.3f}+/-{sub.auc_roc.std():.3f} | macroF1 {sub.macro_f1.mean():.3f}+/-{sub.macro_f1.std():.3f}")
print("Saved -> results/tables/cv_ablation.csv")
print("06_ablation complete.")


fold 0 | A: Fully-connected + lag     macroF1=0.552 AUC=0.581


fold 0 | A: Random graph + lag        macroF1=0.546 AUC=0.479


fold 0 | A: No inter-sensor + lag     macroF1=0.540 AUC=0.529


fold 0 | B: Causal + NO lag           macroF1=0.553 AUC=0.588


fold 0 | C: Causal + lag, EM-only     macroF1=0.507 AUC=0.653


...

fold 4 | A: Random graph + lag        macroF1=0.562 AUC=0.602


fold 4 | A: No inter-sensor + lag     macroF1=0.549 AUC=0.579


fold 4 | B: Causal + NO lag           macroF1=0.564 AUC=0.583


fold 4 | C: Causal + lag, EM-only     macroF1=0.476 AUC=0.514


fold 4 | C: Causal + lag, SM-only     macroF1=0.596 AUC=0.649

Ablation summary (mean +/- std over folds):
  A: Fully-connected + lag       AUC 0.690+/-0.114 | macroF1 0.597+/-0.051
  A: Random graph + lag          AUC 0.676+/-0.141 | macroF1 0.600+/-0.049
  A: No inter-sensor + lag       AUC 0.679+/-0.123 | macroF1 0.594+/-0.049
  B: Causal + NO lag             AUC 0.706+/-0.122 | macroF1 0.620+/-0.063
  C: Causal + lag, EM-only       AUC 0.673+/-0.098 | macroF1 0.568+/-0.073
  C: Causal + lag, SM-only       AUC 0.706+/-0.113 | macroF1 0.635+/-0.073
  Proposed: Causal + lag, all    AUC 0.709+/-0.123 | macroF1 0.622+/-0.066
Saved -> results/tables/cv_ablation.csv
06_ablation complete.
